In [41]:
import pandas as pd
import datetime

In [42]:
team_df = pd.read_csv('data/nfl_team_data.csv')
print(team_df.shape)
team_df.head()

(96, 41)


,Team,Season,Wins,Losses,Tie,O_TotPts,O_Pts/G,O_RushYds,O_RYds/G,O_PassYds,...,O_1downs_Rush,O_1downs_Pass,O_1downs_Pen,O_1downs_Tot,O_3down_Att,O_3down_Made,O_3down_Pct,O_4down_Att,O_4down_Made,O_4down_Pct
0,ARI,2021,11,6,0,449,26.4,2076,122.1,4276,...,127,214,26,367,221,100,45.2,29,17,58.6
1,ATL,2021,7,10,0,313,18.4,1451,85.4,3713,...,75,196,35,306,211,82,38.9,24,10,41.7
2,BAL,2021,8,9,0,387,22.8,2479,145.8,3961,...,159,209,26,394,225,82,36.4,27,18,66.7
3,BUF,2021,11,6,0,483,28.4,2209,129.9,4284,...,134,236,28,398,222,103,46.4,22,11,50.0
4,CAR,2021,5,12,0,304,17.9,1842,108.4,3239,...,118,174,30,322,235,84,35.7,34,14,41.2


In [43]:
all_off_df = pd.read_csv('data/nfl_pass_rush_receive_raw_data.csv')
print(all_off_df.shape)
all_off_df.head()

(19973, 69)


,game_id,player_id,pos,player,team,pass_cmp,pass_att,pass_yds,pass_td,pass_int,...,OT,Roof,Surface,Temperature,Humidity,Wind_Speed,Vegas_Line,Vegas_Favorite,Over_Under,game_date
0,201909050chi,RodgAa00,QB,Aaron Rodgers,GNB,18,30,203,1,0,...,False,outdoors,grass,65,69,10,-3.5,CHI,47.0,2019-09-05
1,201909050chi,JoneAa00,RB,Aaron Jones,GNB,0,0,0,0,0,...,False,outdoors,grass,65,69,10,-3.5,CHI,47.0,2019-09-05
2,201909050chi,ValdMa00,WR,Marquez Valdes-Scantling,GNB,0,0,0,0,0,...,False,outdoors,grass,65,69,10,-3.5,CHI,47.0,2019-09-05
3,201909050chi,AdamDa01,WR,Davante Adams,GNB,0,0,0,0,0,...,False,outdoors,grass,65,69,10,-3.5,CHI,47.0,2019-09-05
4,201909050chi,GrahJi00,TE,Jimmy Graham,GNB,0,0,0,0,0,...,False,outdoors,grass,65,69,10,-3.5,CHI,47.0,2019-09-05


In [44]:
all_def_df = pd.read_csv('data/nfl_dst_raw_data.csv')
print(all_def_df.shape)
all_def_df.head()

(1624, 37)


,game_id,team,def_int,def_int_td,sacks,fumbles_rec,fumbles_rec_td,blocked_kick,safety,def_two_point_conv,...,OT,Roof,Surface,Temperature,Humidity,Wind_Speed,Vegas_Line,Vegas_Favorite,Over_Under,game_date
0,201909050chi,CHI,0,0,5,0,0,0,0,0,...,False,outdoors,grass,65,69,10,-3.5,CHI,47.0,2019-09-05
1,201909050chi,GNB,1,0,5,0,0,0,0,0,...,False,outdoors,grass,65,69,10,-3.5,CHI,47.0,2019-09-05
2,201909080car,LAR,1,0,3,2,0,0,0,0,...,False,outdoors,grass,87,53,3,-1.5,LAR,49.5,2019-09-08
3,201909080car,CAR,1,0,1,0,0,1,0,0,...,False,outdoors,grass,87,53,3,-1.5,LAR,49.5,2019-09-08
4,201909080cle,CLE,0,0,4,0,0,0,0,0,...,False,outdoors,grass,71,55,10,-5.5,CLE,44.0,2019-09-08


In [45]:
def standardize_season_years(df):
    df['game_date'] = pd.to_datetime(df['game_date'], infer_datetime_format=True)
    season = []
    for i,day in enumerate(df['game_date']):
        if  day.month == 1 or day.month == 2:
            season.append(df.iloc[i]['game_date'].year - 1)
        else:
            season.append(day.year)
    df['Season'] = season
    return df

all_def_df = standardize_season_years(all_def_df)
all_off_df = standardize_season_years(all_off_df)

In [46]:
#split out game vs season stats to join with overall team data
team_def = all_def_df[['team', 'def_int', 'def_int_td', 'sacks', 'fumbles_rec',
                       'fumbles_rec_td', 'blocked_kick', 'safety', 'def_two_point_conv',
                       'total_ret_td', 'Season']].copy()
team_off = all_off_df[['team', 'pass_cmp', 'pass_att',
                       'pass_td', 'pass_int', 'pass_sacked', 'pass_sacked_yds',
                       'rush_att', 'rush_td', 'rec', 'rec_yds', 'rec_td',
                       'pass_target_yds', 'pass_poor_throws', 'pass_blitzed', 'pass_hurried', 
                       'rush_yds_before_contact', 'rush_yac','rush_broken_tackles', 'rec_air_yds', 
                       'rec_yac', 'rec_drops','Season']].copy()

In [47]:
team_def_df = team_def.groupby(['Season', 'team']).sum().reset_index()
team_off_df = team_off.groupby(['Season', 'team']).sum().reset_index()
team_off_df.head()

,Season,team,pass_cmp,pass_att,pass_td,pass_int,pass_sacked,pass_sacked_yds,rush_att,rush_td,...,pass_target_yds,pass_poor_throws,pass_blitzed,pass_hurried,rush_yds_before_contact,rush_yac,rush_broken_tackles,rec_air_yds,rec_yac,rec_drops
0,2019,ARI,355,554,20,12,50,320,396,18,...,4127,93,155,63,1289,701,23,3907.2,1889,18
1,2019,ATL,459,684,29,15,50,335,362,10,...,5493,96,217,56,801,560,17,5360.2,1921,17
2,2019,BAL,320,499,38,10,32,145,625,21,...,4485,83,172,33,2063,1418,33,4321.6,1539,21
3,2019,BUF,324,561,22,12,43,274,495,13,...,5114,106,244,49,1100,1126,45,4886.7,1662,40
4,2019,CAR,382,633,17,21,58,484,386,20,...,5275,115,184,76,1205,614,21,5118.3,2058,33


In [48]:
df = pd.merge(team_df, team_off_df,  how='left', left_on=['Team','Season'], right_on = ['team','Season'])
df = pd.merge(df, team_def_df,  how='left', left_on=['Team','Season'], right_on = ['team','Season'])
df.columns

Index(['Team', 'Season', 'Wins', 'Losses', 'Tie', 'O_TotPts', 'O_Pts/G',
       'O_RushYds', 'O_RYds/G', 'O_PassYds', 'O_PYds/G', 'O_TotYds', 'O_Yds/G',
       'D_TotPts', 'D_Pts/G', 'D_RushYds', 'D_RYds/G', 'D_PassYds', 'D_PYds/G',
       'D_TotYds', 'D_Yds/G', 'D_1downs_Rush', 'D_1downs_Pass', 'D_1downs_Pen',
       'D_1downs_Tot', 'D_3down_Att', 'D_3down_Made', 'D_3down_Pct',
       'D_4down_Att', 'D_4down_Made', 'D_4down_Pct', 'O_1downs_Rush',
       'O_1downs_Pass', 'O_1downs_Pen', 'O_1downs_Tot', 'O_3down_Att',
       'O_3down_Made', 'O_3down_Pct', 'O_4down_Att', 'O_4down_Made',
       'O_4down_Pct', 'team_x', 'pass_cmp', 'pass_att', 'pass_td', 'pass_int',
       'pass_sacked', 'pass_sacked_yds', 'rush_att', 'rush_td', 'rec',
       'rec_yds', 'rec_td', 'pass_target_yds', 'pass_poor_throws',
       'pass_blitzed', 'pass_hurried', 'rush_yds_before_contact', 'rush_yac',
       'rush_broken_tackles', 'rec_air_yds', 'rec_yac', 'rec_drops', 'team_y',
       'def_int', 'def_int_td', 's

In [49]:
df.head()

,Team,Season,Wins,Losses,Tie,O_TotPts,O_Pts/G,O_RushYds,O_RYds/G,O_PassYds,...,team_y,def_int,def_int_td,sacks,fumbles_rec,fumbles_rec_td,blocked_kick,safety,def_two_point_conv,total_ret_td
0,ARI,2021,11,6,0,449,26.4,2076,122.1,4276,...,ARI,13.0,1.0,41.0,14.0,1.0,0.0,0.0,0.0,0.0
1,ATL,2021,7,10,0,313,18.4,1451,85.4,3713,...,ATL,12.0,2.0,17.0,8.0,0.0,1.0,0.0,0.0,0.0
2,BAL,2021,8,9,0,387,22.8,2479,145.8,3961,...,BAL,9.0,1.0,33.0,6.0,1.0,1.0,0.0,0.0,0.0
3,BUF,2021,11,6,0,483,28.4,2209,129.9,4284,...,BUF,21.0,1.0,46.0,9.0,0.0,2.0,0.0,0.0,0.0
4,CAR,2021,5,12,0,304,17.9,1842,108.4,3239,...,CAR,9.0,0.0,39.0,6.0,0.0,0.0,0.0,0.0,0.0


In [50]:
df.info()
df.to_csv('data/all_ream_data.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 96 entries, 0 to 95
Data columns (total 73 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Team                     96 non-null     object 
 1   Season                   96 non-null     int64  
 2   Wins                     96 non-null     int64  
 3   Losses                   96 non-null     int64  
 4   Tie                      96 non-null     int64  
 5   O_TotPts                 96 non-null     int64  
 6   O_Pts/G                  96 non-null     float64
 7   O_RushYds                96 non-null     int64  
 8   O_RYds/G                 96 non-null     float64
 9   O_PassYds                96 non-null     int64  
 10  O_PYds/G                 96 non-null     float64
 11  O_TotYds                 96 non-null     int64  
 12  O_Yds/G                  96 non-null     float64
 13  D_TotPts                 96 non-null     int64  
 14  D_Pts/G                  96 

In [51]:
df = df[['Team', 'Season', 'Wins', 'Losses', 'O_Pts/G', 'O_RYds/G', 'O_PYds/G', 
         'O_1downs_Tot', 'O_3down_Pct', 'rush_att', 'rush_td', 
         'pass_att', 'pass_td','pass_int', 'pass_sacked', 'pass_sacked_yds', 'pass_poor_throws', 'rec_drops',
         'D_Pts/G', 'D_RYds/G', 'D_PYds/G', 'D_1downs_Tot', 'D_3down_Pct', 
         'def_int', 'def_int_td', 'sacks','fumbles_rec', 'fumbles_rec_td', 
         'blocked_kick', 'safety', 'total_ret_td']]
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 96 entries, 0 to 95
Data columns (total 31 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Team              96 non-null     object 
 1   Season            96 non-null     int64  
 2   Wins              96 non-null     int64  
 3   Losses            96 non-null     int64  
 4   O_Pts/G           96 non-null     float64
 5   O_RYds/G          96 non-null     float64
 6   O_PYds/G          96 non-null     float64
 7   O_1downs_Tot      96 non-null     int64  
 8   O_3down_Pct       96 non-null     float64
 9   rush_att          96 non-null     int64  
 10  rush_td           96 non-null     int64  
 11  pass_att          96 non-null     int64  
 12  pass_td           96 non-null     int64  
 13  pass_int          96 non-null     int64  
 14  pass_sacked       96 non-null     int64  
 15  pass_sacked_yds   96 non-null     int64  
 16  pass_poor_throws  96 non-null     int64  
 17 

In [52]:
df.to_csv('data/all_team_data_trimmed.csv', index=False)